# Day 14: Pandas 进阶 —— 字符串方法、apply、缺失值、分箱

> **目标**: 掌握 Pandas 数据清洗的高级技巧，理解 `.str` 访问器和 `.apply` 的适用场景。
> **前置**: Day 13 的 DataFrame 基础（loc/iloc/筛选/类型转换）
> **数据**: `../data/sales.csv`（500 行，9 列）

## 1. `.str` 访问器 —— 字符串操作不用循环

Pandas 的每一列都可以调用 `.str` 来批量做字符串操作，和 Python 字符串方法类似，但作用于整列。

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sales.csv")

# 转大写 / 小写
print(df["country"].str.upper().head())

# 包含某个子串（模糊匹配）
has_lap = df["product"].str.contains("lap", case=False)
print(f"含 'lap' 的产品: {has_lap.sum()} 个")

# 以某字符串开头 / 结尾
o_orders = df["order_id"].str.startswith("O10")
print(f"O10 开头的订单: {o_orders.sum()} 个")

# 替换字符串
print(df["country"].str.replace("US", "USA").unique())

# 切片（取前3个字符）
print(df["customer_id"].str[:3].unique())

# 长度
print(df["product"].str.len().head())

0    GERMANY
1         US
2         US
3         US
4     FRANCE
Name: country, dtype: object
含 'lap' 的产品: 87 个
O10 开头的订单: 100 个
['Germany' 'USA' 'France' 'UK' 'China']
['C00']
0     8
1     8
2     6
3    10
4     5
Name: product, dtype: int64


## 2. `.apply` —— 对每行/每列做自定义操作

`apply` 是 Pandas 的「万能胶水」，但性能比向量化慢。原则是：**能向量化就不 apply**。

In [2]:
# 对 Series 的每个元素 apply（不推荐，但要知道）
# 例：根据 total 给订单打标签
def label_order(total):
    if total >= 5000:
        return "高"
    elif total >= 1000:
        return "中"
    return "低"

df["order_label"] = df["total"].apply(label_order)
print(df["order_label"].value_counts())

# 更优雅的做法：用 np.where（向量化，更快）
df["order_label2"] = np.where(
    df["total"] >= 5000, "高",
    np.where(df["total"] >= 1000, "中", "低")
)
print(df["order_label2"].value_counts())

# 对 DataFrame 的每行 apply（axis=1）
# 例：计算每个订单的「单价」和「折扣率」
def calc_metrics(row):
    return pd.Series({
        "unit_price": row["total"] / row["quantity"],
        "discount_ratio": 1 - row["total"] / (row["price"] * row["quantity"])
    })

metrics = df.apply(calc_metrics, axis=1)
print(metrics.head())

# 注意：apply 慢，大数据量优先用向量化
# 上面的 metrics 可以用向量化重写：
df["unit_price_vec"] = df["total"] / df["quantity"]
df["discount_vec"] = 1 - df["total"] / (df["price"] * df["quantity"])

order_label
中    242
低    174
高     84
Name: count, dtype: int64
order_label2
中    242
低    174
高     84
Name: count, dtype: int64
   unit_price  discount_ratio
0      1299.0             0.0
1        99.0             0.0
2        99.0             0.0
3        99.0             0.0
4        99.0             0.0


## 3. 缺失值处理 —— 数据清洗必会

数据岗最常见的脏数据就是缺失值。策略：**先判断，再处理**。

In [3]:
# 判断缺失值
print(df.isnull().sum())           # 每列缺失值数量
print(df["total"].isnull().any()) # 某列是否有缺失

# 删除缺失值（简单粗暴）
df_clean = df.dropna()             # 删除包含任何 NaN 的行
df_clean2 = df.dropna(subset=["total", "quantity"])  # 只删指定列有 NaN 的行

# 填充缺失值（更常用）
df_fill = df.copy()
df_fill["total"] = df_fill["total"].fillna(0)        # 填 0
df_fill["total"] = df_fill["total"].fillna(df_fill["total"].mean())  # 填均值
df_fill["country"] = df_fill["country"].fillna("Unknown")  # 填字符串

# 前向填充 / 后向填充（时间序列常用）
df_fill["total"] = df_fill["total"].fillna(method="ffill")   # 用前一个值填充
df_fill["total"] = df_fill["total"].fillna(method="bfill")   # 用后一个值填充

# 条件填充：根据其他列的值决定填充策略
df_fill.loc[df_fill["country"] == "UK", "total"] = df_fill.loc[df_fill["country"] == "UK", "total"].fillna(1000)

order_id          0
customer_id       0
product           0
category          0
quantity          0
price             0
order_date        0
country           0
total             0
order_label       0
order_label2      0
unit_price_vec    0
discount_vec      0
dtype: int64
False


C:\Users\69261\AppData\Local\Temp\ipykernel_9952\2849550746.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_fill["total"] = df_fill["total"].fillna(method="ffill")   # 用前一个值填充
C:\Users\69261\AppData\Local\Temp\ipykernel_9952\2849550746.py:17: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_fill["total"] = df_fill["total"].fillna(method="bfill")   # 用后一个值填充


## 4. 分箱 —— cut 与 qcut

把连续数值分成离散区间，是数据分组的常用技巧。

In [4]:
# pd.cut —— 按等宽区间分箱
df["total_bin"] = pd.cut(
    df["total"],
    bins=[0, 1000, 3000, 6000, 10000],
    labels=["低", "中", "高", "超高"]
)
print(df["total_bin"].value_counts())

# pd.qcut —— 按等分位数分箱（每箱数量大致相等）
df["total_q"] = pd.qcut(df["total"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df["total_q"].value_counts())

# 分箱后可以做分组统计
print(df.groupby("total_bin")["quantity"].mean())

total_bin
中     188
低     174
高      90
超高     48
Name: count, dtype: int64
total_q
Q1    141
Q2    130
Q4    125
Q3    104
Name: count, dtype: int64
total_bin
低     2.344828
中     2.861702
高     3.444444
超高    4.750000
Name: quantity, dtype: float64


C:\Users\69261\AppData\Local\Temp\ipykernel_9952\1508448424.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("total_bin")["quantity"].mean())


## 5. 条件赋值 —— where / mask

`np.where` 的三元表达式写法，在 Pandas 中也可以用 `where` 和 `mask` 方法。

In [5]:
# where: 满足条件保留原值，不满足用指定值替代
df["total_capped"] = df["total"].where(df["total"] <= 5000, 5000)
# 等价于: total if total <= 5000 else 5000

# mask: where 的反面，满足条件用指定值替代，不满足保留原值
df["total_capped2"] = df["total"].mask(df["total"] > 5000, 5000)

# 对比 np.where（最灵活）
df["total_capped3"] = np.where(df["total"] > 5000, 5000, df["total"])

print(df[["total", "total_capped", "total_capped2", "total_capped3"]].head(10))

   total  total_capped  total_capped2  total_capped3
0   2598          2598           2598           2598
1     99            99             99             99
2    396           396            396            396
3    396           396            396            396
4    495           495            495            495
5    198           198            198            198
6   6495          5000           5000           5000
7   1198          1198           1198           1198
8   1495          1495           1495           1495
9    297           297            297            297


## 6. 简单透视表 —— pivot_table

透视表 = 按行列维度做统计汇总，是 Excel 的核心功能。

In [6]:
# 按 country 和 category 统计 total 的总和
pivot = pd.pivot_table(
    df,
    values="total",
    index="country",
    columns="category",
    aggfunc="sum",
    fill_value=0
)
print(pivot)

# 多聚合函数
pivot2 = pd.pivot_table(
    df,
    values="total",
    index="country",
    aggfunc=["sum", "mean", "count"],
    fill_value=0
)
print(pivot2)

category  Accessory  Audio  Computer  Mobile
country                                     
China         21271   8191     32159   18480
France        54935  60245     51431   41554
Germany       45042  18975     51036   28967
UK           164595  73734    158700  137788
US           155423  43541     70607   31042
            sum         mean count
          total        total total
country                           
China     80101  2503.156250    32
France   208165  2602.062500    80
Germany  144020  2182.121212    66
UK       534817  2714.807107   197
US       300613  2404.904000   125


## 今日要点总结

| 操作 | 代码 | 适用场景 | 性能 |
|------|------|----------|------|
| `.str.contains` | 模糊匹配子串 | 筛选含关键词的行 | 快 |
| `.str.startswith/endswith` | 前缀/后缀匹配 | 筛选 ID 前缀 | 快 |
| `.str.replace` | 批量替换 | 数据标准化 | 快 |
| `.str.len` | 字符串长度 | 验证字段长度 | 快 |
| `.apply` | 自定义函数 | 复杂逻辑无法用向量化 | 慢 |
| `np.where` | 条件赋值 | 简单 if-else 映射 | 快 |
| `.fillna` | 填充缺失 | 数据补全 | 快 |
| `.dropna` | 删除缺失 | 脏数据清理 | 快 |
| `pd.cut` | 等宽分箱 | 按数值范围分组 | 快 |
| `pd.qcut` | 等分位分箱 | 按分布分箱 | 快 |
| `.where` | 条件保留 | 上限/下限截断 | 快 |
| `pivot_table` | 二维透视 | 行列统计汇总 | 快 |

**核心心法**: 字符串操作用 `.str`，缺失值先判断再处理，复杂逻辑用 `apply`（但优先向量化），分组统计用 `pivot_table` 或 `groupby`。